# LLM で学習データを合成して小さな分類器を学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/llm_synthetic_data.ipynb)

NLP クックブック 12-17 ／ テキスト分類

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

人手のラベル付きデータがほとんどない。手元の LLM に (1) ラベルごとの例文を書かせる、(2) ラベルのない文書にラベルを付けさせる、の 2 通りで学習データを作り、軽い分類器（TF-IDF + ロジスティック回帰、文埋め込み + ロジスティック回帰）を学習したい。人手のラベルが各クラス 10 件ある場合と比べて、合成データの量と精度の関係や、生成した文の偏りも確かめる。

（T4 の GPU ランタイムで、モデルのダウンロードを含めて全体で 15〜20 分程度かかる。）

In [1]:
import io, random, tarfile, urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

def load_livedoor():
    """livedoor ニュースコーパス（株式会社ロンウイット配布、記事は CC BY-ND 2.1 JP: 改変禁止）を DataFrame にする"""
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
pool, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=0)
LABELS = sorted(df["label"].unique())
# 記事タイトルで 9 クラスに分類する。人手のラベル付きデータは各クラス 10 件。選び方（乱数の種）を 5 通り変える
SEEDS, N = range(5), 10
human = {s: pool.groupby("label").sample(N, random_state=s) for s in SEEDS}
used = set().union(*[set(d.index) for d in human.values()])
unlabeled = pool.drop(list(used)).sample(1500, random_state=0)    # ラベルのないテキストの代わり（正解は分析にだけ使う）
print("人手:", len(human[0]), "件 × 5 通り / ラベルなし:", len(unlabeled), "件 / テスト:", len(test), "件")

# ラベルの説明（LLM に渡す）。カテゴリは配信元のサイトなので、サイトの内容を短く説明する
DESC = {
    "dokujo-tsushin": "独身女性の恋愛・結婚・仕事・生活に関するコラム",
    "it-life-hack": "パソコン、インターネット、ソフトウェア、IT 製品の使いこなし",
    "kaden-channel": "家電製品やデジタル機器、ガジェットの話題",
    "livedoor-homme": "大人の男性向けの仕事、キャリア、ライフスタイル",
    "movie-enter": "映画の公開情報、俳優、監督、試写会",
    "peachy": "女性向けのグルメ、美容、ファッション、エンタメ",
    "smax": "スマートフォンや携帯電話、モバイル通信",
    "sports-watch": "野球やサッカーなどスポーツの試合や選手",
    "topic-news": "芸能人やネットで話題の出来事、社会のニュース",
}

人手: 90 件 × 5 通り / ラベルなし: 1500 件 / テスト: 1474 件


In [2]:
# 手元の LLM: SB Intuitions の sarashina2.2-3b-instruct（MIT ライセンス）
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

BF16 = torch.cuda.get_device_capability()[0] >= 8      # T4 は bf16 が使えないので float16
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tok = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tok.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
tok.padding_side = "left"                               # バッチで生成するので左パディング
lm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat_prompt(msg):
    return tok.apply_chat_template([{"role": "user", "content": msg}], add_generation_prompt=True, tokenize=False)

In [3]:
# 方法 1: ラベルごとに見出しを生成させる。(a) ラベルの説明だけを渡す、(b) 人手の 10 件から 5 件を例として見せる
# (b) は人手のデータの選び方（種）ごとに、その種の 10 件だけを例にして作る（比べる相手の人手 10 件以外の人手のデータが混ざらないように）
# (a) 各クラス 100 件 + (b) 各クラス 80 件 × 5 通り = 4,500 件。Colab の T4 で 6〜9 分
def gen_prompt(label, examples=None):
    msg = f"ニュースサイトの「{label}」カテゴリ（{DESC[label]}）に載りそうな記事の見出しを 1 つ考えてください。"
    if examples:
        msg += "\n\nこのカテゴリの見出しの例:\n" + "\n".join(f"- {e}" for e in examples) + "\n\n例とは違う話題で、新しい見出しを書いてください。"
    return chat_prompt(msg + "見出しだけを 1 行で出力してください。")

def clean(s):
    s = s.strip().split("\n")[0].strip().lstrip("-・ ")
    for p in ["見出し:", "見出し："]:
        s = s.removeprefix(p).strip()
    return s[1:-1] if s.startswith("「") and s.endswith("」") else s

@torch.no_grad()
def generate(prompts, batch_size=64):
    out = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(lm.device)
        gen = lm.generate(**enc, max_new_tokens=48, do_sample=True, temperature=1.0, top_p=0.95, pad_token_id=tok.pad_token_id)
        out += [clean(s) for s in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return out

rng = random.Random(0)
torch.manual_seed(0)
rows = [("説明だけ", -1, label, gen_prompt(label)) for label in LABELS for _ in range(100)]
for s in SEEDS:
    for label in LABELS:
        seed_titles = list(human[s].query("label == @label")["title"])
        rows += [("例 5 件", s, label, gen_prompt(label, rng.sample(seed_titles, 5))) for _ in range(80)]
gen_df = pd.DataFrame(rows, columns=["kind", "seed", "label", "prompt"])
gen_df["title"] = generate(list(gen_df["prompt"]))
gen_df = gen_df[gen_df["title"].str.len() > 0].drop(columns="prompt")    # 空の出力を捨てる
synth = {k: d.reset_index(drop=True) for k, d in gen_df.groupby("kind")}
for kind, d in synth.items():
    print(f"--- {kind}（{len(d)} 件。例 5 件は種 0 の分を表示）")
    for label in ["sports-watch", "smax", "dokujo-tsushin"]:
        print(f"[{label}]", " / ".join(d.query("label == @label and seed <= 0")["title"].head(4)))
print(f"GPU メモリの最大: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

--- 例 5 件（3600 件。例 5 件は種 0 の分を表示）
[sports-watch] テニスの全米オープン、Novak Djokovicの試合放棄に憶測飛び交う：健康問題それとも戦略か / 東京オリンピック金メダリスト、岩崎恭子の新しいコーチ就任と今後の展望に迫る / テニス全仏オープン、日本人ペアが悲願のベスト8進出—新たな時代への第一歩 / 大谷翔平、新人王ダブル受賞の可能性が高まる新記録達成へ
[smax] 次世代モバイル通信規格5G、ドコモが国内初商用化へ向けた計画を発表 / 次世代5G対応スマートフォン、ソニーエリクソンから革新的なデザイン・スペックで来月登場か / 次世代モバイル通信規格5G対応デバイス、2023年に登場予測：主要メーカーの開発動向を追う / 5G時代の到来：携帯キャリアが推進するeスポーツ大会の開催と技術検証
[dokujo-tsushin] 独身女性必見、最新 fintech ツールで変わる家計簿管理と資産運用 / AI時代における独女の仕事観：専門性と柔軟性のバランス / 令和の独女が選ぶ、本当に幸せな結婚へと導く恋愛テクニック5選 / コロナ禍を経て進化するシングル女性のライフスタイル：新しい趣味や価値観の変化を徹底調査
--- 説明だけ（900 件。例 5 件は種 0 の分を表示）
[sports-watch] 新記録樹立! 伝説のプレイヤー、引退試合で圧巻のパフォーマンス / 新記録樹立！ チームAの大谷翔平、史上最高の投手兼打者パフォーマンスを披露 / 新記録樹立！伝説の打者が挑む、史上最大規模のプロ野球マッチ / 新記録樹立！期待の若手投手がデビュー戦で完全試合を達成 - sports-watchが独占報道
[smax] 新作5Gスマホ発売、AI機能が大幅に進化 - ユーザー体験が劇的に向上すること間違いなし / 新機種発表直前：smax ユーザーが選ぶべき最新スマホの特徴とは / 新機種発表間近？大手メーカーが次世代折りたたみ式スマートフォンを公開テストへ / 新発表の5G対応スマホ、 speeds と省電力性で業界に驚きをもたらす
[dokujo-tsushin] 最新研究が明かす、独身女性が幸せなキャリアと恋愛を両立させるコツ / 最新結婚トレンド解析：独身女性が選ぶパートナーの特徴とは？ / 最新婚活ト

In [4]:
# 方法 2: ラベルのない見出しに LLM でラベルを付ける（疑似ラベル）。9 つの選択肢を番号で示し、次のトークンが「1」〜「9」になる確率を比べる
# （大規模言語モデルの章の「選択肢の尤度を比べて LLM に分類させる」と同じ考え方）。テストデータも同じ方法で分類して、LLM そのものの正解率も見る
# ラベルなし 1,500 件 + テスト 1,474 件。Colab の T4 で 2〜3 分
menu = "\n".join(f"{k + 1}. {DESC[l]}" for k, l in enumerate(LABELS))
choice_ids = [tok.convert_tokens_to_ids(str(k + 1)) for k in range(len(LABELS))]   # 「1」〜「9」はそれぞれ 1 トークン

@torch.no_grad()
def llm_classify(titles, batch_size=32):
    probs = []
    for i in range(0, len(titles), batch_size):
        prompts = [chat_prompt(f"次のニュース記事の見出しは、どのカテゴリの記事ですか。\n\n{menu}\n\n見出し: {t}\n\n番号だけで答えてください。")
                   for t in titles[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(lm.device)
        logits = lm(**enc, logits_to_keep=1).logits[:, -1, choice_ids].float()   # 最後の位置（左パディングなのでそろう）の 9 つの番号のロジットだけ
        probs.append(logits.softmax(-1).cpu().numpy())
    return np.concatenate(probs)

P_unl = llm_classify(list(unlabeled["title"]))
P_test = llm_classify(list(test["title"]))
L = np.array(LABELS)
print(f"LLM で直接テストを分類した正解率: {(L[P_test.argmax(1)] == test['label']).mean():.3f}")
print("ラベルなしの文書に付いたラベルの件数:", dict(zip(LABELS, np.bincount(P_unl.argmax(1), minlength=len(LABELS)).tolist())))
# 特定の番号を選びやすい偏りを、ラベルなしの文書での確率の平均で割って弱める（正解ラベルは使わない）
prior = P_unl.mean(0)
P_unl, P_test = P_unl / prior, P_test / prior
P_unl, P_test = P_unl / P_unl.sum(1, keepdims=True), P_test / P_test.sum(1, keepdims=True)
print(f"偏りを補正したあとのテストの正解率: {(L[P_test.argmax(1)] == test['label']).mean():.3f}")
pseudo = unlabeled.assign(label=L[P_unl.argmax(1)], conf=P_unl.max(1), gold=unlabeled["label"])
print(f"疑似ラベルの正解率（ラベルなし 1,500 件の正解と比べる。本来は分からない）: {(pseudo['label'] == pseudo['gold']).mean():.3f}")
print(f"GPU メモリの最大: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

LLM で直接テストを分類した正解率: 0.399
ラベルなしの文書に付いたラベルの件数: {'dokujo-tsushin': 35, 'it-life-hack': 31, 'kaden-channel': 178, 'livedoor-homme': 30, 'movie-enter': 40, 'peachy': 204, 'smax': 314, 'sports-watch': 25, 'topic-news': 643}
偏りを補正したあとのテストの正解率: 0.425
疑似ラベルの正解率（ラベルなし 1,500 件の正解と比べる。本来は分からない）: 0.455
GPU メモリの最大: 10.6 GB


In [5]:
# 人手・合成・疑似ラベルで学習し、同じテストデータで正解率を比べる（分類器は TF-IDF + LR と、文埋め込み + LR）
# 14 条件 × 5 通り × 2 分類器 = 140 回の学習。Colab で 1〜2 分
import gc
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

lm = None; gc.collect(); torch.cuda.empty_cache()   # LLM の GPU メモリを空ける（生成や疑似ラベルをやり直すときは、LLM の読み込みのセルから）
st = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")   # MIT ライセンス
all_texts = sorted(set(df["title"]) | {t for d in synth.values() for t in d["title"]})
vec = dict(zip(all_texts, st.encode(["query: " + t for t in all_texts], normalize_embeddings=True, batch_size=256)))
embed = lambda texts: np.stack([vec[t] for t in texts])
X_test = embed(test["title"])

def make_lr():
    return LogisticRegression(C=10, solver="liblinear")    # 何度も学習し直すので、速い liblinear（クラスごとの 2 値分類）を使う

def evaluate(d):
    tfidf = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(1, 3), sublinear_tf=True), make_lr())
    acc1 = tfidf.fit(d["title"], d["label"]).score(test["title"], test["label"])
    acc2 = make_lr().fit(embed(d["title"]), d["label"]).score(X_test, test["label"])
    return acc1, acc2

def per_class(d, n, s):     # 各クラス n 件を選ぶ（足りなければ全部）
    return pd.concat([g.sample(min(n, len(g)), random_state=s) for _, g in d.groupby("label")])

def synth_ex(s):            # 種 s の人手 10 件を例にして生成したもの
    return synth["例 5 件"].query("seed == @s")

def agreed(s):   # 人手 10 件で学習した分類器の予測と LLM の疑似ラベルが一致したものだけを、人手 10 件に足す
    pred = make_lr().fit(embed(human[s]["title"]), human[s]["label"]).predict(embed(pseudo["title"]))
    return pd.concat([human[s], pseudo[pred == pseudo["label"]]])

conditions = {
    "人手 各 10 件": lambda s: human[s],
    "人手 各 50 件": lambda s: pool.groupby("label").sample(50, random_state=s),
    "人手 各 200 件": lambda s: pool.groupby("label").sample(200, random_state=s),
    **{f"生成（説明だけ）各 {n} 件": (lambda s, n=n: per_class(synth["説明だけ"], n, s)) for n in [10, 40, 80]},
    **{f"生成（例 5 件）各 {n} 件": (lambda s, n=n: per_class(synth_ex(s), n, s)) for n in [10, 40, 80]},
    "人手 10 + 生成（例 5 件）各 80 件": lambda s: pd.concat([human[s], per_class(synth_ex(s), 80, s)]),
    "疑似ラベル 全部": lambda s: pseudo,
    "疑似ラベル 確信度の上位半分": lambda s: pseudo[pseudo["conf"] >= pseudo["conf"].median()],
    "人手 10 + 疑似ラベル（全部）": lambda s: pd.concat([human[s], pseudo]),
    "人手 10 + 疑似ラベル（分類器と一致）": agreed,
}

acc = {cond: np.array([evaluate(make(s)) for s in SEEDS]) for cond, make in conditions.items()}
rows = []
for cond, a in acc.items():
    row = {"学習データ": cond, "件数（種 0）": len(conditions[cond](0))}
    for j, clf_name in enumerate(["TF-IDF", "埋め込み"]):
        diff = a[:, j] - acc["人手 各 10 件"][:, j]       # 同じ種どうしの差（人手 10 件を足す条件では、同じ人手のデータとの対応のある比較）
        half = stats.t.ppf(0.975, len(diff) - 1) * diff.std(ddof=1) / np.sqrt(len(diff))
        row[f"{clf_name} 正解率"] = f"{a[:, j].mean():.3f} ± {a[:, j].std(ddof=1):.3f}"
        row[f"{clf_name} 人手 10 件との差 [95%CI]"] = f"{diff.mean():+.3f} [{diff.mean() - half:+.3f}, {diff.mean() + half:+.3f}]"
    rows.append(row)
pd.set_option("display.width", 200)
print(pd.DataFrame(rows).to_string(index=False))

                  学習データ  件数（種 0）    TF-IDF 正解率 TF-IDF 人手 10 件との差 [95%CI]      埋め込み 正解率 埋め込み 人手 10 件との差 [95%CI]
              人手 各 10 件       90 0.604 ± 0.015   +0.000 [+0.000, +0.000] 0.644 ± 0.005 +0.000 [+0.000, +0.000]
              人手 各 50 件      450 0.737 ± 0.014   +0.132 [+0.098, +0.167] 0.743 ± 0.003 +0.099 [+0.094, +0.104]
             人手 各 200 件     1800 0.815 ± 0.003   +0.210 [+0.191, +0.230] 0.793 ± 0.004 +0.149 [+0.138, +0.159]
         生成（説明だけ）各 10 件       90 0.300 ± 0.010   -0.304 [-0.323, -0.286] 0.505 ± 0.007 -0.139 [-0.154, -0.124]
         生成（説明だけ）各 40 件      360 0.357 ± 0.015   -0.247 [-0.276, -0.218] 0.521 ± 0.011 -0.123 [-0.135, -0.111]
         生成（説明だけ）各 80 件      720 0.392 ± 0.014   -0.212 [-0.247, -0.177] 0.523 ± 0.001 -0.121 [-0.128, -0.114]
        生成（例 5 件）各 10 件       90 0.388 ± 0.031   -0.216 [-0.257, -0.176] 0.539 ± 0.013 -0.105 [-0.125, -0.085]
        生成（例 5 件）各 40 件      360 0.446 ± 0.004   -0.158 [-0.176, -0.140] 0.562 ± 0.005 -0.082 [-0.088, -0.076]
 

In [6]:
# 生成した見出しの多様性と偏り: 同じ件数（各クラス 80 件）の本物の見出しと比べる（例 5 件は種 0 の分）
from collections import Counter

real = per_class(pool.drop(list(used)), 80, 0)
corpus_titles = set(df["title"])

def stats_of(d, generated=True):
    sims, d2 = [], []
    for _, g in d.groupby("label"):
        e = embed(g["title"])
        s = e @ e.T
        sims.append(s[np.triu_indices(len(g), 1)].mean())                      # 同じクラスの中の 2 件の類似度の平均
        bigrams = [t[i:i + 2] for t in g["title"] for i in range(len(t) - 1)]
        d2.append(len(set(bigrams)) / len(bigrams))                            # distinct-2（文字 2-gram の異なり / 延べ）
    return {"件数": len(d), "完全重複の割合": 1 - d["title"].nunique() / len(d), "distinct-2": np.mean(d2),
            "クラス内の類似度": np.mean(sims), "平均の長さ": d["title"].str.len().mean(),
            "コーパスの見出しと一致": d["title"].isin(corpus_titles).sum() if generated else np.nan}

sets = {"本物": real, "生成（説明だけ）": per_class(synth["説明だけ"], 80, 0), "生成（例 5 件）": per_class(synth_ex(0), 80, 0)}
display(pd.DataFrame({k: stats_of(v, k != "本物") for k, v in sets.items()}).T.round(3))

# 生成した見出しでよく使われる言い回し（sports-watch の先頭 4 文字の頻度と、最も多いものの例）
for k, d in sets.items():
    titles = d.query("label == 'sports-watch'")["title"]
    heads = Counter(t[:4] for t in titles).most_common(5)
    print(f"{k}:", ", ".join(f"{h}({c})" for h, c in heads))
    print("    例:", " / ".join(titles[titles.str.startswith(heads[0][0])].head(2)))

,件数,完全重複の割合,distinct-2,クラス内の類似度,平均の長さ,コーパスの見出しと一致
本物,720.0,0.001,0.666,0.826,37.154,NaN
生成（説明だけ）,720.0,0.001,0.373,0.887,38.338,0.0
生成（例 5 件）,720.0,0.000,0.461,0.868,40.446,0.0


本物: 【Spo(45), 日本代表(2), なでしこ(2), 楽天退団(1), 止まぬ紗(1)
    例: 【Sports Watch】パラグアイ戦終了、中西氏「マン・オブ・ザ・マッチは、本田」 / 【Sports Watch】巨人1位の澤村、涙の理由
生成（説明だけ）: 新記録樹(45), 大谷翔平(5), MLB新(4), メジャー(4), プロ野球(3)
    例: 新記録樹立か？！注目投手の歴史的イニング無失点試合を堪能せよ / 新記録樹立！日本代表サッカーチームが国際大会で初の金メダルを獲得
生成（例 5 件）: 【Spo(14), 大谷翔平(6), テニス全(3), 新テニス(3), tenn(2)
    例: 【Sports Watch】新ルール導入で変わるテニス界、ロジャー・フェデラーの戦略と不安 / 【Sports Watch】大谷翔平、新記録目指す二刀流シーズンへの意気込みと課題を語る


In [7]:
# 生成した見出しのラベルは正しいか: 人手 10 件と無関係な本物の見出し（約 5,800 件）で学習した分類器を「審判」にして確かめる
# （分析のためだけの審判。実際にラベルが少ない場面では使えない）
judge = LogisticRegression(C=10, max_iter=1000).fit(embed(pool.drop(list(used))["title"]), pool.drop(list(used))["label"])
print(f"審判のテストでの正解率: {judge.score(X_test, test['label']):.3f}")
for k, d in synth.items():
    pred = judge.predict(embed(d["title"]))
    agree = pd.Series(pred == d["label"]).groupby(d["label"]).mean()
    print(f"生成（{k}）: 審判と生成時のラベルが一致 {np.mean(pred == d['label']):.1%}（クラス別: 最低 {agree.idxmin()} {agree.min():.0%}、最高 {agree.idxmax()} {agree.max():.0%}）")

# 疑似ラベルの偏り: LLM が付けたラベルの件数と本当の件数、確信度ごとの正解率
print(pd.DataFrame({"疑似ラベルの件数": pseudo["label"].value_counts(), "本当の件数": pseudo["gold"].value_counts(),
                    "クラス別の適合率": (pseudo["label"] == pseudo["gold"]).groupby(pseudo["label"]).mean().round(3)}).to_string())
q = pd.qcut(pseudo["conf"], 4, labels=["下位 1/4", "2", "3", "上位 1/4"])
print("確信度の 4 分位ごとの正解率:", (pseudo["label"] == pseudo["gold"]).groupby(q, observed=True).mean().round(3).to_dict())

審判のテストでの正解率: 0.820
生成（例 5 件）: 審判と生成時のラベルが一致 76.6%（クラス別: 最低 kaden-channel 40%、最高 movie-enter 98%）
生成（説明だけ）: 審判と生成時のラベルが一致 74.9%（クラス別: 最低 topic-news 29%、最高 movie-enter 99%）
                疑似ラベルの件数  本当の件数  クラス別の適合率
dokujo-tsushin        43    180     0.930
it-life-hack          54    165     0.611
kaden-channel        175    189     0.423
livedoor-homme        47     86     0.489
movie-enter           77    183     0.662
peachy               198    166     0.581
smax                 282    187     0.553
sports-watch          41    175     0.634
topic-news           583    169     0.281
確信度の 4 分位ごとの正解率: {'下位 1/4': 0.325, '2': 0.357, '3': 0.477, '上位 1/4': 0.659}


- LLM に学習データを作らせ、推論の軽い小さなモデルに学ばせる方法は、ZeroGen（Ye ら, EMNLP 2022）などで研究されている。この章の「言い換えたデータを足して少ない学習データを補う（データ拡張）」が手元の文を言い換えるのに対し、方法 1 は文そのものを新しく作り、方法 2 は LLM を「ラベル付け関数」の 1 本として使う（この章の「ルールでラベルを付けて分類器を学習する（弱教師あり学習、Snorkel）」と同じ位置づけ）。データ拡張のレシピと同じく、タイトルで 9 クラス、人手は各クラス 10 件、選び方を 5 通り変えて比べた。例を見せる生成は、種ごとにその種の人手 10 件だけを例にした（別の種の人手のデータを例に使うと、実質的に人手のデータが増え、人手 10 件との比較にならない）。
- ラベルの説明だけで生成した見出しは、各クラス 80 件（計 720 件）使っても正解率が TF-IDF で 0.392、埋め込みで 0.523 で、人手 10 件 (0.604 / 0.644) に届かなかった。埋め込みでは 40 件からほとんど伸びない (0.521 → 0.523)。人手の例を 5 件見せて生成すると 0.464 / 0.571 に上がったが、それでも人手 10 件より 14.1 / 7.3 ポイント低い。人手 10 件に足しても、TF-IDF では +0.019（95% 信頼区間 −0.012〜+0.049）で改善ははっきりせず、埋め込みでは −0.032 下がった。人手を各クラス 50 件にしたとき (+0.132 / +0.099) にはまったく及ばない。なお、この信頼区間は人手のデータの選び方（例 5 件の生成では、種ごとの生成も含む）のばらつきだけを表す。テストデータ、説明だけの生成文、疑似ラベルは 1 回しか作っていないので、疑似ラベルの条件の正解率は種によらず (± 0.000)、その信頼区間は人手 10 件の側のばらつきだけを表す。
- 生成した見出しは、完全に重複したものはほとんどない（0.1% 以下で、本物と同程度）のに、各クラス 80 件で比べると文字 2-gram の異なりの割合 (distinct-2) は本物の 0.666 に対して 0.373（説明だけ）/ 0.461（例 5 件）と低く、同じクラスの中の埋め込みの類似度も高い（本物 0.826 に対して 0.887 / 0.868）。説明だけで作った sports-watch の 80 件のうち 45 件が「新記録樹」（「新記録樹立！」など）で始まり、例を見せると本物の「【Sports Watch】」という書き出しまでまねた。重複率だけでは多様性の低さは見えないので、n-gram の異なりや言い回しの頻度も見る。また、コーパスの記事は大半が 2010〜2012 年のものなのに、生成文には 5G、令和、コロナ禍、大谷翔平の二刀流など後の時代の話題や、英語の混じった見出し（Novak Djokovic、fintech など）が出てくる。コーパスの見出しとそのまま一致したものはなかった。
- 生成時のラベルが正しいかを、本物の約 5,800 件で学習した審判の分類器（テストで 0.820）で見ると、一致は 7 割台（説明だけ 74.9%、例 5 件 76.6%）で、topic-news（説明だけで 29%）や kaden-channel（例 5 件で 40%）が特に低かった。このコーパスのラベルは配信元のサイトなので、家電・IT・スマートフォンや、何でも載るトピックニュースの境目は、短い説明からは決まらない。Li ら (EMNLP 2023) も、合成データで学習した分類器は主観的な課題ほど本物のデータとの差が大きく、生成文の多様性も低いと報告している。
- ここでは、疑似ラベルで学習した分類器 (0.375 / 0.376) は、LLM で直接テストを分類した正解率（0.399、確率の平均で割って偏りを補正すると 0.425）より低かった。ラベルなしの 1,500 件の疑似ラベルの正解率は 0.455 で、偏りを補正したあとでも topic-news が 583 件（補正前は 643 件、本当は 169 件）に膨らみ、sports-watch は 41 件（本当は 175 件）しかなかった。確信度の上位半分に絞ると（上位 1/4 の正解率は 0.659 と高い）かえって 0.342 に下がり、人手 10 件の分類器と一致したものだけを人手 10 件に足しても、人手 10 件だけより 4.6 / 7.3 ポイント低い。疑似ラベルを使う前に、少量の人手のラベル付きデータで LLM の正解率を測り、人手 10 件の分類器より明らかに高いことを確かめる。この 3B のモデルとこのラベルの組み合わせでは、合成データだけで学習するより人手で 10 件ずつラベルを付けるほうが効き、合成データを人手の 10 件に足してもはっきりした改善はなかった。
- LLM の出力で別のモデルを学習してよいかは、モデルや API の規約で決まる（2026 年 11 月に確かめたもの）。ここで使った sarashina2.2-3b-instruct-v0.1 は MIT ライセンスで、MIT ライセンスには出力の利用を制限する条項はない。OpenAI の利用規約は「OpenAI と競合するモデルの開発に出力を使う」ことを、Anthropic の商用利用規約（2025 年 6 月 17 日発効）は「競合する製品やサービスを作るためにサービスを使う（競合する AI モデルの学習を含む）」ことを、Gemini API の追加利用規約（2026 年 3 月 23 日発効）は「サービスと競合するモデルの開発にサービスを使う」ことを禁じている。オープンなモデルでも、Llama 3.1 のライセンスは出力で学習したモデルを配布するなら名前の先頭に「Llama」を付けることを求め、Gemma の利用規約は、Gemma と同じように動作させる目的で Gemma の合成データを使って学習（蒸留）したモデルを「Model Derivatives」に含めて規約の対象にしている（出力そのものは Model Derivatives ではないとしている）。小さな分類器が「競合」にあたるかは解釈の問題なので、業務で使う前に最新の規約を読み、必要なら法務に確認する。
- livedoor ニュースコーパスは CC BY-ND 2.1 JP（改変禁止）。人手の見出しを例としてプロンプトに入れているので、生成したデータや疑似ラベルを付けたデータは手元の実験だけに使い、配布しない。`intfloat/multilingual-e5-small` は MIT ライセンス。sarashina2.2 のトークナイザーの修正は、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。GPU メモリの最大は 10.6 GB（生成のバッチ 64 件）で、T4 の 15GB に収まる。